# B1.1 · Historical parsing and structural indexing

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.0 · Start here — what an AI SDLC means](https://spbreed.github.io/cyber-commons/lessons/B1.0.html)**.

| | |
|---|---|
| Open-source tooling | git, OpenGrep, tree-sitter |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Haiku 4.5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

An agent with a two-million-token context and a four-million-line repository has the same problem as an analyst with a week: it cannot read everything, so the question is what it reads first. Structure is what makes that choice something other than luck.

## 2 · The framework

```
   4,000,000 lines            context budget: ~30,000 lines
   +-------------------+      +----------------+
   |  the repository   | ---> |  what it reads |
   +-------------------+      +----------------+
              |
        structure decides the arrow

   symbol graph . call edges . entrypoints . change history
   -> "start at the functions reachable from an HTTP handler and changed
      in the last year" is a choice. "the first 30k lines" is not.
```

Most review starts at the diff. That is the smallest possible context and it
throws away the single best predictor you have: **this repository has already
told you where it breaks.**

Phase 1 of the pipeline fixes that, and it begins with two stages that run
before any analysis:

**Stage 1 — Historical parsing.** Extract prior vulnerabilities, the commits
that fixed them, and pull-request history. Files that have been fixed for
security reasons before are dramatically more likely to be fixed again. This is
one of the oldest empirical results in software engineering and almost nobody
wires it into a scanner.

**Stage 2 — Structural indexing.** Break the codebase into *semantic units* —
functions, classes, modules — and index how they relate. Not lines, not files.
A scanner that reasons over lines cannot answer "who calls this?", and every
later stage needs that answer.

Together these produce the two inputs the rest of the pipeline runs on: a
**risk-ranked file list** and a **structural index**.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 1 — historical parsing

A realistic slice of repository history: commits, their subjects, and which of them were security fixes.

In [ ]:
import re, math
from dataclasses import dataclass, field
from collections import Counter, defaultdict

@dataclass(frozen=True)
class Commit:
    sha: str; subject: str; files: tuple; days_ago: int

HISTORY = [
 Commit("a1b2c3d", "fix(auth): reject empty session tokens (CVE-2025-0091)",
        ("src/auth.py", "src/session.py"), 420),
 Commit("b2c3d4e", "refactor: extract render helper", ("src/render.py",), 400),
 Commit("c3d4e5f", "fix(billing): SQL injection in report filter (CVE-2025-1188)",
        ("src/billing.py",), 300),
 Commit("d4e5f6a", "feat: add CSV export", ("src/billing.py", "src/export.py"), 260),
 Commit("e5f6a7b", "security: patch path traversal in doc fetch",
        ("src/docs.py",), 210),
 Commit("f6a7b8c", "chore: bump deps", ("requirements.txt",), 180),
 Commit("a7b8c9d", "fix(auth): timing leak in token compare",
        ("src/auth.py",), 150),
 Commit("b8c9d0e", "feat: pagination on reports", ("src/billing.py",), 120),
 Commit("c9d0e1f", "fix: harden docs path join after report", ("src/docs.py",), 60),
 Commit("d0e1f2a", "style: formatting", ("src/render.py", "src/export.py"), 30),
]

SECURITY_MARKERS = re.compile(
    r"\b(cve-\d{4}-\d+|security|injection|traversal|xss|ssrf|auth|hardcoded|"
    r"leak|sanitis|sanitiz|escap)\w*", re.I)

def is_security_fix(c):
    return bool(SECURITY_MARKERS.search(c.subject))

sec = [c for c in HISTORY if is_security_fix(c)]
print(f"{len(HISTORY)} commits, {len(sec)} security-relevant\n")
for c in sec:
    print(f"   {c.sha}  {c.days_ago:>4}d  {c.subject[:56]}")
    print(f"{'':14s}touched {list(c.files)}")

In [ ]:
def risk_zones(history, half_life_days=180):
    """Prior-defect density, decayed by age. Recent security fixes weigh more."""
    score = defaultdict(float)
    fixes = defaultdict(int)
    churn = Counter()
    for c in history:
        for f in c.files:
            churn[f] += 1
            if is_security_fix(c):
                fixes[f] += 1
                score[f] += math.exp(-c.days_ago / half_life_days)
    rows = []
    for f in churn:
        rows.append({"file": f, "commits": churn[f], "security_fixes": fixes[f],
                     "risk": round(score[f], 3)})
    return sorted(rows, key=lambda r: -r["risk"])

zones = risk_zones(HISTORY)
print(f"{'file':22s}{'commits':>9}{'sec fixes':>11}{'risk':>8}")
print("-" * 52)
for r in zones:
    print(f"{r['file']:22s}{r['commits']:>9}{r['security_fixes']:>11}{r['risk']:>8}")
print("\nsrc/auth.py and src/docs.py are the repeat zones. Nothing has been")
print("scanned yet — this ordering comes entirely from history.")

## 4 · Stage 2 — structural indexing

Now index the code into semantic units. `ast` does the real work here; in a polyglot repo this is what tree-sitter is for.

In [ ]:
import ast

SOURCES = {
 "src/auth.py": '''
def compare_token(supplied, stored):
    return supplied == stored

def login(request):
    user = lookup(request["user"])
    if user and compare_token(request["token"], user.token):
        return make_session(user)
    return None
''',
 "src/billing.py": '''
def build_filter(owner):
    return "WHERE owner = '" + owner + "'"

def list_reports(conn, owner):
    return conn.execute("SELECT * FROM reports " + build_filter(owner))
''',
 "src/docs.py": '''
def safe_join(base, name):
    return base + "/" + name

def fetch(base, name):
    return open(safe_join(base, name)).read()
''',
}

@dataclass
class Unit:
    name: str; file: str; line: int; calls: tuple; params: tuple

def index(sources):
    units, by_name = [], {}
    for path, src in sources.items():
        tree = ast.parse(src)
        for fn in [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]:
            calls = tuple(sorted({
                (c.func.id if isinstance(c.func, ast.Name) else
                 getattr(c.func, "attr", ""))
                for c in ast.walk(fn) if isinstance(c, ast.Call)} - {""}))
            u = Unit(fn.name, path, fn.lineno, calls,
                     tuple(a.arg for a in fn.args.args))
            units.append(u); by_name[fn.name] = u
    return units, by_name

units, by_name = index(SOURCES)
print(f"{'unit':16s}{'file':16s}{'line':>5}  params → calls")
print("-" * 74)
for u in units:
    print(f"{u.name:16s}{u.file:16s}{u.line:>5}  {list(u.params)} → {list(u.calls)}")

## 5 · Where it breaks — an index of files cannot answer the question

The whole point of semantic units is the relationships between them. Here is the question every later stage asks, and what each kind of index can say about it.

In [ ]:
def callers_of(name, units):
    return [u.name for u in units if name in u.calls]

QUESTION = "who reaches safe_join(), and with what?"
print(QUESTION)
print(f"   line-based index : cannot answer — 'safe_join' appears in 2 places")
print(f"   file-based index : 'it is in src/docs.py'")
print(f"   semantic index   : callers = {callers_of('safe_join', units)}, "
      f"reached from fetch(base, name)")

reverse = {u.name: callers_of(u.name, units) for u in units}
print("\nreverse call index:")
for name, callers in reverse.items():
    print(f"   {name:16s}← {callers or '(entry point)'}")
entry_points = [n for n, c in reverse.items() if not c]
print(f"\nentry points (nothing calls them): {entry_points}")

In [ ]:
# Stage 1 + Stage 2 combined: the pipeline's actual input.
def phase1_partial(history, sources):
    zones = {r["file"]: r["risk"] for r in risk_zones(history)}
    units, _ = index(sources)
    out = []
    for u in units:
        out.append({"unit": u.name, "file": u.file,
                    "historical_risk": zones.get(u.file, 0.0),
                    "params": list(u.params), "calls": list(u.calls)})
    return sorted(out, key=lambda r: -r["historical_risk"])

pipeline_input = phase1_partial(HISTORY, SOURCES)
print(f"{'unit':16s}{'file':16s}{'hist risk':>11}")
print("-" * 44)
for r in pipeline_input:
    print(f"{r['unit']:16s}{r['file']:16s}{r['historical_risk']:>11.3f}")

top = pipeline_input[0]["file"]
assert top == "src/auth.py", top
print(f"\nAnalysis budget goes to {top} first — decided before a single rule ran.")
print("Note why docs.py ranks below it: the recent 'harden docs path join' commit")
print("does not match the security markers, so it scores nothing. Marker quality")
print("is the whole accuracy of stage 1, and it is worth tuning on your own history.")

## What you just proved

Four of ten commits match the security markers. `src/auth.py` ranks highest on decayed risk (0.53) — two dated security fixes, the more recent dominating — followed by `src/docs.py` (0.31) and `src/billing.py` (0.19), purely from history. Note that a recent 'harden docs path join' commit scores nothing because it matches no marker. The structural index extracts six functions with their parameters and calls, the reverse index identifies `login`, `list_reports` and `fetch` as entry points, and the combined Phase 1 output orders units by historical risk before any scanning.

## Your turn

Run the stage-1 query against a real repository: `git log --name-only --grep='CVE\|security\|injection'`. Rank the files by how often they appear. That list usually surprises people, and it is free.

---

**Next → [B1.2 · Component summarisation and architecture synthesis](https://spbreed.github.io/cyber-commons/lessons/B1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*